# Fit Models

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.multiclass import OneVsRestClassifier

from tqdm_joblib import tqdm_joblib
from tqdm.auto import tqdm

from bioacoustics.data import load_results
from bioacoustics.metrics import evaluate_multilabel_model

from bioacoustics.models import ignore_warnings
ignore_warnings() # TODO: fix the reasons for these warnings

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Problems to think about:

- Multi-label prediction -> hierarchical models? implement with `Pipeline`
- Class imbalance -> tinker with class weights, adjust probabilities?
- Use additional metadata from train, not available in soundscapes
- Maybe use hybrid approaches? learn about how to combine different models' predictions

think about chunking the audio, tuning the threshold for ech class

## Load features and labels

In [8]:
data_train = load_results("features", "data_train")
data_train_soundscapes = load_results("features", "data_train_soundscapes")

Note that it doesn't make much sense to validate on iNat or XC data, since it's of different format than test soundscapes.

In [21]:
LABEL_NAME = ("y_class", "y_primary")[0]
TRAIN_ON_SOUNDSCAPES = True

if TRAIN_ON_SOUNDSCAPES:
    X = data_train["X"]
    Y = data_train[LABEL_NAME]
    X_train, X_test, y_train, y_test = train_test_split(
        X, Y, test_size=0.2, random_state=42
    )
else: 
    X_train = data_train["X"]
    X_test = data_train_soundscapes["X"]
    y_train = data_train[LABEL_NAME]
    y_test = data_train_soundscapes[LABEL_NAME]


## Predict class

Start with predicting class only, we may after use this result when predicting primary label (hierarchical approach).



In [22]:
# clf = LogisticRegression(solver="liblinear", max_iter=1000, class_weight="balanced")
# clf = RandomForestClassifier(
#     n_estimators=200, max_depth=None, n_jobs=-1, class_weight="balanced"
# )
clf = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="logloss",
)

In [24]:
# TODO: try XGBoost, RF etc, change multi-label strategies
# TODO: hierarchical approach - train a model per class
# and predict species conditioned on class (can try soft versions - multiply probas)

# TODO: alternative to hierarchical approach - include predicted class as a feature
# for species prediction
# TODO: the same idea for training metadata that is absent for the test
# - we can learn to predict this metadata as a secondary task and then include
# it in the model
# NOTE: attention to data leak between train - validation when using secondary tasks
# to generate features (class, metadata)

# TODO: can use this metadata as well for stratification - better split validation
# (make sure that there is no data leak because of the same site)
# TODO: check whether metadata alone can predict species (check for shortcut learning)

# TODO: smart cross validations

# TODO: quality of train audio from iNat and xeno-cant is poorer and further from test than train soundscapes,
#       so maybe we should use them only for species poorly covered by soundscapes

# TODO: impute some NaNs

# TODO: maybe stratify be species or sth else since train data had
#   too (really) many bird recordings, whereas soundscapes contain more amphibians

# TODO: maybe play with thresholds

# TODO: temporal smoothing of predicted probabilities!

pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),  # otherwise MFCC and RMS are incomparable
        (
            "clf",
            OneVsRestClassifier(clf, n_jobs=-2, verbose=2),
        ),
    ]
)
with tqdm_joblib(
    tqdm(
        desc="Training OvR", total=y_train.shape[1]
    )
):
    pipeline.fit(X_train, y_train)

Training OvR:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

[Parallel(n_jobs=-2)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=-2)]: Done   3 out of   5 | elapsed:   10.4s remaining:    6.9s
[Parallel(n_jobs=-2)]: Done   5 out of   5 | elapsed:   13.3s finished


In [25]:
y_pred, y_proba = evaluate_multilabel_model(pipeline, X_test, y_test)


================= CLASSIFICATION REPORT =================
              precision    recall  f1-score   support

    Amphibia       1.00      0.17      0.29        65
        Aves       0.99      1.00      0.99      6472
     Insecta       0.96      0.57      0.72        40
    Mammalia       1.00      0.08      0.14        13
    Reptilia       0.00      0.00      0.00         0

   micro avg       0.99      0.99      0.99      6590
   macro avg       0.79      0.36      0.43      6590
weighted avg       0.99      0.99      0.98      6590
 samples avg       0.99      0.99      0.99      6590


================= THRESHOLD-BASED METRICS =================
Macro F1:   0.4289
Micro F1:   0.9870
Hamming loss: 0.0052

================= RANKING & PROBABILITY METRICS =================
Macro ROC AUC: nan
Micro ROC AUC: 0.9994
Macro AP:      0.5319
Micro AP:      0.9979
LRAP:          0.9935


## Interpret the model

In [26]:
# TODO: get feature importance